# **RL4AA'26 Challenge:** SAC

This notebook uses the [SAC](https://stable-baselines3.readthedocs.io/en/master/modules/sac.html) implementation of _Stable Baselines3_ to solve the steering and tuning task.

If you are more experienced, feel free to change the algorithm or move to a completely different RL library altogether, provided you believe that it will help you solve the challenge.

## IMPORTANT: PLEASE READ BEFORE RUNNING THE NOTEBOOK

Training a good RL agent can take a lot of time ⏰

The task in this challenge was designed such that 100,000 `total_timesteps` is enough to solve it well. 

On a Mac with an Apple-made SoC the training can be performed in **10-20 minutes**. 

Expect at least **15 minutes** on a recent Windows or Linux laptop per training run.

More importantly this training 🚨 **can use a considerable amount of RAM** 🚨, meaning your computer may crash when training your agent locally. So please **make sure that you have saved your work before you start training**. If you have access to a computing cluster you can send your training runs there.

Your RAM will scale with the rollout size `n_envs` x `n_steps`

## Make sure you have followed the setting-up instructions in this repository's README


In [ ]:
import math
from functools import partial
from pathlib import Path
import platform

import numpy as np
import torch.nn as nn
from rl_zoo3 import linear_schedule
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv

import wandb
from src.eval import Study
from src.eval.eval_rl_v3_sim import evaluate_policy
from src.train.clara_ppo import make_env
from src.environments.clara import TransverseTuning
from src.utils import load_config, log_challenge_results, save_config, save_reward_source

# Set your team name!
Pick a unique name for your team! (mandatory). 

This is how you will be identified in the Weights and Biases leaderboard.

In [ ]:
MyTeamName = ""

# Shape your reward!
You can find a basic reward below to get you started.

Create more reward classes below with a unique name and test them.

In [ ]:
"""
BasicReward is a simple example reward function.

If you use any other imported functions than
    import math
    import numpy as np
    from src.environments.clara import TransverseTuning
you must also add those imports to the REWARD_EXTRA_IMPORTS list below, if you want
them saved alongside your reward class (not necessary for challenge).
"""

REWARD_EXTRA_IMPORTS = []  # e.g. ["from scipy.special import expit"]


class BasicReward(TransverseTuning):
    def _get_reward(self) -> float:
        """
        Computes the reward for the current step.

        You can make use of the following information to compute the reward:
         - self.backend.get_beam_parameters(): Returns a NumPy array with the current
              beam parameters (mu_x, sigma_x, mu_y, sigma_y).
            - self._target_beam: NumPy array with the target beam parameters (mu_x,
                sigma_x, mu_y, sigma_y).
            - self.backend.is_beam_on_screen(): Boolean indicating whether the beam is
                on the screen.
            - self.backend.get_magnets(): NumPy array with the current magnet settings
                as (k1_Q1, k1_Q2, angle_CV, k1_Q3, angle_CH).
            - self._previous_magnet_settings: NumPy array with the magnet settings
                before the current action was taken as (angle_CV, angle_CH, k1_Q1,
                k1_Q2, k1_Q3).

        You are allowed to make use of any other information available in the
        environment and backend, if you are so inclined to look through the code.

        Tips: rewards must always be finite, avoid division by zero, avoid NaNs,
        consider using e.g. `np.clip` to bound values, etc.
        """
        if not self.backend.is_beam_on_screen():
            self._on_screen_reward = -100.0
            return self._on_screen_reward

        dist = np.linalg.norm(self.backend.get_beam_parameters() - self._target_beam)

        r = -math.log(dist)

        # Track separate reward components for logging and analysis, but only return the final combined reward `r` to the agent.
        self._beam_reward = r
        self._on_screen_reward = 0.0
        return r

Design your own reward function by creating a new reward class below, and then run the rest of the notebook to train your model with that reward.

In [ ]:
"""
Modify this class to implement a custom reward for your model.

This cell will be saved as a separate file, and stored alongside your trained models,
allowing you to check what custom reward was used for a given model.

If you use any other imported functions than
    import math
    import numpy as np
    from src.environments.clara import TransverseTuning
you must also add those imports to the REWARD_EXTRA_IMPORTS list below, if you want
them saved alongside your reward class (not necessary for challenge).
"""


class MyCustomReward(TransverseTuning):

    def _get_reward(self) -> float:
        """
        Computes the reward for the current step.

        Available information:
        - self.backend.get_beam_parameters(): [mu_x, sigma_x, mu_y, sigma_y]
        - self._target_beam: Target beam parameters [mu_x, sigma_x, mu_y, sigma_y]
        - self.backend.is_beam_on_screen(): Boolean
        - self.backend.get_magnets(): [angle_CV, angle_CH, k1_Q1, k1_Q2, k1_Q3]
        - self._previous_magnet_settings: Previous magnet settings
        """

        #####################
        # To be implemented!
        #####################
        reward = -1

        return reward

## Don't forget to choose your reward here

In [ ]:
# reward = MyCustomReward
reward = BasicReward

## Configure the RL training of your SAC algorithm from _Stable Baselines3_

While the choice of your reward is expected to be the major contributor to your final models success, further tuning of hyperparameters could likely increase your performance. You could make your model train faster and reach higher performance more efficiently by tuning learning rates or algorithm gamma

Below you will find the configuration for training your SAC agent, split into two sections:

1. **TUNE THESE** - The hyperparameters you should experiment with to improve your agent. This includes environment settings (action mode, magnet limits, episode length), SAC algorithm parameters (learning rate, batch size, discount factor, etc.), and policy network architecture. For some of the parameters, a comment like [1.0 - 5.0] is added after: this is referencing a reasonable tuning range to give you an idea of the types of values you are expected to try (but you are allowed to put whatever you want).

2. **DO NOT CHANGE** - System and infrastructure settings that should be left at their defaults unless you really know what you are doing. Can be adapted to fit your system in case you have very low/high amounts of RAM for faster training.

A third section of parameters that are **fixed during evaluation** is included at the bottom but should not be modified.

**Note:** During evaluation, your agent will be tested on 150-step episodes (not the 50-step default used in training), with random incoming beams and misalignments.

In [ ]:
# =============================================================================
# TUNE THESE - Experiment with these to improve your agent's performance
# =============================================================================
tuning_config = {
    # ----- Environment (CLARA simulation) -----
    # Maximum allowed quadrupole setting strengt (real range is -30 to 30, but lower
    # values constrain the agent to more physically reasonable settings)
    "max_quad_setting": 30.0,  # [10.0 - 30.0]
    # Max change per step for quadrupoles.
    # Higher values allow the agent more freedom but may lead to unstable behavior (the agent can "overshoot" good settings).
    "max_quad_delta": 2.0,  # [1.0 - 5.0]
    # Max change per step for steerers
    "max_steerer_delta": 6.1782e-4,  # [3e-4, 12e-4]
    # Magnet initialisation on reset: None (keep current), "random", or an
    # array of 5 values. The default is a focusing-defocusing-focusing pattern.
    "magnet_init_mode": np.array([0.0, 0.0, 10.0, -10.0, 10.0]),
    # Steps per training episode (evaluation uses 150 steps, but shorter episodes can speed up training)
    "max_episode_steps": 50,
    # Number of past observations stacked together (1 = current only)
    "frame_stack": 1,
    # ----- SAC algorithm -----
    # See: https://stable-baselines3.readthedocs.io/en/master/modules/sac.html
    "batch_size": 256,
    "learning_starts": 100,
    "learning_rate": 3e-4,  # [3e-5 - 3e-3]
    "lr_schedule": "constant",  # "constant" or "linear"
    "buffer_size": 1_000_000,
    "tau": 0.005,
    "gamma": 0.99,
    "ent_coef": "auto",  # "auto" or a float value like 0.1 (higher values encourage more exploration)
    # ----- Policy network -----
    # "small" = [64,64], "medium" = [256,256] (original SAC paper)
    "net_arch": "medium",
    "activation_fn": "ReLU",  # "Tanh", "ReLU", or "GELU"
    "log_std_init": -3,
    "clip_mean": 2.0,
    "use_sde": False,
    "sde_sample_freq": -1,
}

# =============================================================================
# DO NOT CHANGE - Unless you know what you are doing.
# (n_envs can be scaled according to your system)
# =============================================================================
system_config = {
    # Observation and action scaling (required for the policy to work correctly)
    "normalize_observation": True,
    "rescale_action": True,
    # Number of parallel environments and gradient steps per env step.
    # More environments speed up data collection but use more RAM.
    # USING TOO LARGE VALUES MAY CAUSE OTHER APPLICATIONS TO CRASH!
    "n_envs": 10,
    "n_steps": 1,
    # PyTorch device for training ("auto", "cpu", or "cuda")
    "sb3_device": "auto",
    # "dummy" = sequential environments, "subproc" = separate processes
    "vec_env": "subproc",
    # Host details (for monitoring only)
    # Feel free to replace with empty string if you prefer.
    "host_os": platform.system(),
    "host_os_release": platform.release(),
    "host_cpu": platform.processor(),
    "reward_name": reward.__name__,
    "team_name": MyTeamName,
    "algo": "SAC",
}

# =============================================================================
# FIXED FOR EVALUATION - These are overridden during evaluation, do not modify
# =============================================================================
_fixed_config = {
    # "direct" sets magnets to absolute values; "delta" adjusts by increments
    "action_mode": "delta",
    "total_timesteps": 100_000,  # Total environment steps for training fixed for challenge, but can be experimented with on your own.
    "incoming_mode": "random",
    "misalignment_mode": "random",
    "max_misalignment": 5e-4,
    "simulate_finite_screen": False,
    "target_beam_mode": np.zeros(4),
    "target_threshold": None,
    "clip_magnets": True,
}

## Train your RL agent

Below is the code for training the RL agent with the SAC algorithm as implemented in _Stable Baselines3_. You probably shouldn't change this cell unless you know what you are doing. If you want to use a different algorithm, you will need to change the code below.


In [ ]:
# =============================================================================
# DO NOT CHANGE - Just execute the cell to start the training and track it
# in weights and biases through our leaderboard.
# =============================================================================
if system_config["team_name"] == "":
    raise ValueError(
        "You must set a team name first! You can set one at the top of the notebook."
    )
# Setup wandb
wandb.init(
    entity="rl4aa26-challenge",
    project="challenge-leaderboard",
    sync_tensorboard=True,  # if you want to save some RAM you can set this to False, but then you won't see training curves until the end of training
    monitor_gym=True,
    config=system_config
    | _fixed_config,  # intentionally exclude tuning_config from challenge logging
    dir=".wandb",
    tags=[],
)
config = dict(wandb.config)
config = config | tuning_config
config["run_name"] = wandb.run.name

# Setup vectorised environments for training and evaluation
assert config["vec_env"] in ["dummy", "subproc"]
vec_env = (
    DummyVecEnv([partial(make_env, config, reward) for _ in range(config["n_envs"])])
    if config["vec_env"] == "dummy"
    else SubprocVecEnv(
        [partial(make_env, config, reward) for _ in range(config["n_envs"])]
    )
)
eval_vec_env = DummyVecEnv(
    [
        partial(
            make_env,
            config,
            reward,
            plot_episode=False,  # Disable plotting for evaluation to speed it up, set to True if you want to see the beam during evaluation episodes.
            show_rewards_only=True,
            log_task_statistics=True,
        )
    ]
)

# Setup learning rate schedule if needed
if config["lr_schedule"] == "linear":
    config["learning_rate"] = linear_schedule(config["learning_rate"])

# Setup RL training algorithm
model = SAC(
    policy="MlpPolicy",
    env=vec_env,
    learning_rate=config["learning_rate"],
    buffer_size=config["buffer_size"],
    learning_starts=config["learning_starts"],
    batch_size=config["batch_size"],
    tau=config["tau"],
    gamma=config["gamma"],
    n_steps=config["n_steps"],
    ent_coef=config["ent_coef"],
    use_sde=config["use_sde"],
    sde_sample_freq=config["sde_sample_freq"],
    policy_kwargs={
        "activation_fn": getattr(nn, config["activation_fn"]),
        "net_arch": {  # From rl_zoo3
            "small": {"pi": [64, 64], "qf": [64, 64]},
            "medium": {"pi": [256, 256], "qf": [256, 256]},
        }[config["net_arch"]],
        "clip_mean": config["clip_mean"],
        "log_std_init": config["log_std_init"],
    },
    device=config["sb3_device"],
    tensorboard_log=f"log/sac/{config['run_name']}",
    verbose=0,  # Set to 1 to print info in the jupyter notebook during training
)

# Setup callbacks for evaluation and logging
eval_callback = EvalCallback(
    eval_vec_env,
    eval_freq=1_000,
    n_eval_episodes=5,
    best_model_save_path=f"models/clara/sac/{wandb.run.name}/best_model",
)

# Store configuration and custom reward environment
Path(f"models/clara/sac/{wandb.run.name}").mkdir(parents=True, exist_ok=True)
save_config(config, f"models/clara/sac/{wandb.run.name}/config")
save_reward_source(
    reward,
    f"models/clara/sac/{wandb.run.name}",
    extra_imports=REWARD_EXTRA_IMPORTS,
)

# Train the model
model.learn(
    total_timesteps=config["total_timesteps"],
    callback=[eval_callback],
)

# Save the final model with associated configuration.
# The best evaluated model is saved automatically by the
# eval_callback, and stored in models/clara/sac/{wandb.run.name}/best_model/best_model.zip
model.save(f"models/clara/sac/{wandb.run.name}/model")

# =============================================================================
# Your trained agents are evaluated automatically, and the best score 
# is sent to the leaderboard!
#
# Both the best model (from EvalCallback) and the final model are evaluated
# separately, and the better of the two is used for the leaderboard score.
# =============================================================================
model_name = config["run_name"]
model_path = Path("models") / "clara" / "sac" / model_name
config = load_config(model_path / "config")

model_best_evaluated = SAC.load(model_path / "best_model/best_model", device=config["sb3_device"])
model_final = SAC.load(model_path / "model", device=config["sb3_device"])

evaluate_policy(model_best_evaluated, config, write_data=True, seed=42, data_dir=f"data/{model_name}_best")
evaluate_policy(model_final, config, write_data=True, seed=42, data_dir=f"data/{model_name}_final")

study_best = Study.load(f"data/{model_name}_best", name=model_name)
study_final = Study.load(f"data/{model_name}_final", name=model_name)

challenge_score, best_model_is, best_eval_model_score, final_model_score = log_challenge_results(study_best, study_final)

print("======== FINAL RESULTS ========")
print(f"Best-evaluated model score:      {best_eval_model_score:.4f}")
print(f"Final model score:               {final_model_score:.4f}")
print(f"Challenge score (best of above): {challenge_score:.4f}  (--> Submitted to Leaderboard!)")

# Cleanup environments
vec_env.close()
eval_vec_env.close()

wandb.run.finish()

# Watch your agent by plotting your episodes

In [ ]:
# Plot episodes from the model that achieved the challenge score
study = study_best if best_model_is == "best_evaluated" else study_final
# Uncomment one of the lines below to plot episodes from either model separately if you prefer.
# study = study_best
# study = study_final

episodes_to_plot = 5
for i in range(episodes_to_plot):
    _ = study.episodes[i].plot_summary()

## Re-evaluate your agent (optional)

If you want to re-run the evaluation independently (e.g. after restarting the kernel), use the cell below. Note that scores won't automatically be sent to WandB, and if you have already evaluated the models once you could simplet load the studies instead.

In [ ]:
# =============================================================================
# Optional: Re-evaluate a previously trained model
# =============================================================================

# Could be set manually as well, e.g. model_name = "leafy-star-1"
model_name = config["run_name"] 

model_path = Path("models") / "clara" / "sac" / model_name
config = load_config(model_path / "config")

model_best_evaluated = SAC.load(model_path / "best_model/best_model", device=config["sb3_device"])
model_final = SAC.load(model_path / "model", device=config["sb3_device"])

print("Calculating score for best model from EvalCallback...")
evaluate_policy(model_best_evaluated, config, write_data=True, seed=42, data_dir=f"data/{model_name}_best")
print("--------------------------------------------------------")
print("Calculating score for final model after full training...")
evaluate_policy(model_final, config, write_data=True, seed=42, data_dir=f"data/{model_name}_final")

study_best = Study.load(f"data/{model_name}_best", name=model_name)
study_final = Study.load(f"data/{model_name}_final", name=model_name)

print("Evaluating best-evaluated model...")
best_eval_model_score = study_best.evaluate_challenge()
print("--------------------------------")
print("Evaluating final model...")
final_model_score = study_final.evaluate_challenge()
challenge_score = min(best_eval_model_score, final_model_score)

print("======== FINAL RESULTS ========")
print(f"Best-evaluated model score: {best_eval_model_score:.4f}")
print(f"Final model score:          {final_model_score:.4f}")
print(f"Challenge score:            {challenge_score:.4f}")